In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsentregafinal")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = (
    f"abfss://{container}@{storageName}.dfs.core.windows.net/"
    "inventario/inventario_raw.csv"
)

tabla_destino = f"{catalogo}.{esquema}.inventario"

print(f"Ruta de origen: {ruta}")
print(f"Tabla de destino: {tabla_destino}")

In [0]:
df_inventario_read = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("inferSchema", "false")
    .csv(ruta)
)

In [0]:
df_inventario_read.printSchema()

display(df_inventario_read)

In [0]:
df_inventario_read = df_inventario_read.toDF(
    *[
        columna.strip().lower()
        for columna in df_inventario_read.columns
    ]
)

print(df_inventario_read.columns)

In [0]:
columnas_tabla = [
    campo.name
    for campo in spark.table(tabla_destino).schema.fields
    if campo.name.lower() != "ingestion_date"
]

columnas_tabla_normalizadas = [
    columna.lower()
    for columna in columnas_tabla
]

print("Columnas del archivo:")
print(df_inventario_read.columns)

print("\nColumnas de la tabla Bronze:")
print(columnas_tabla)

In [0]:
columnas_archivo = set(df_inventario_read.columns)
columnas_esperadas = set(columnas_tabla_normalizadas)

columnas_faltantes = columnas_esperadas - columnas_archivo
columnas_adicionales = columnas_archivo - columnas_esperadas

if columnas_faltantes:
    raise ValueError(
        f"El archivo no contiene estas columnas requeridas: "
        f"{sorted(columnas_faltantes)}"
    )

if columnas_adicionales:
    print(
        "Advertencia: el archivo contiene columnas adicionales "
        f"que no se cargarán: {sorted(columnas_adicionales)}"
    )

print("Las columnas requeridas coinciden correctamente.")

In [0]:
df_inventario_final = (
    df_inventario_read
    .select(
        *[
            col(columna.lower())
            .cast("string")
            .alias(columna)
            for columna in columnas_tabla
        ]
    )
    .withColumn(
        "ingestion_date",
        current_timestamp()
    )
)

In [0]:
df_inventario_final.printSchema()

display(df_inventario_final)

In [0]:
df_inventario_final.write.mode("overwrite").insertInto(
    tabla_destino
)

In [0]:
display(
    spark.table(tabla_destino)
)

In [0]:
cantidad_origen = df_inventario_read.count()

cantidad_bronze = (
    spark.table(tabla_destino)
    .count()
)

print(f"Registros leídos del archivo: {cantidad_origen}")
print(f"Registros cargados en Bronze: {cantidad_bronze}")

if cantidad_origen != cantidad_bronze:
    raise ValueError(
        "La cantidad de registros del origen no coincide con Bronze."
    )

print("Carga de inventario completada correctamente.")